# Extra biosphere and iLUC import

This notebook imports the additional biosphere flows first and then imports the iLUC database.

Project names, database names and file paths are defined in `functions/utils.py`.
The preparation and linking functions are stored in `functions/iluc_import.py`.


## 1. Setup background

In [1]:
import bw2data as bd
import pandas as pd

from functions.fun_iluc_import import (
    prepare_extra_biosphere_import,
    prepare_iluc_import,
)
from functions.utils import Config

In [2]:
bd.projects.set_current(Config.PROJECT_NAME)

print(f"Current Brightway project: {bd.projects.current}")

Current Brightway project: project_name


## 2. Check input files

Both Excel workbooks are stored in the shared OneDrive database folder.


In [3]:
assert Config.EXTRA_BIOSPHERE_PATH.is_file(), (
    f"Extra biosphere workbook not found: {Config.EXTRA_BIOSPHERE_PATH}"
)

assert Config.ILUC_PATH.is_file(), (
    f"iLUC workbook not found: {Config.ILUC_PATH}"
)

print(f"Extra biosphere: {Config.EXTRA_BIOSPHERE_PATH}")
print(f"iLUC workbook:    {Config.ILUC_PATH}")

Extra biosphere: C:\Users\SimonVemmelund\OneDrive - 2.-0 LCA Consultants ApS\Databases\iLUC_BW\extra_biosphere.xlsx
iLUC workbook:    C:\Users\SimonVemmelund\OneDrive - 2.-0 LCA Consultants ApS\Databases\iLUC_BW\iLUC_for_bw_ei312_conseq.xlsx


## 3. Prepare extra biosphere import

The extra biosphere is prepared first because the iLUC database may contain elementary flows that link to it.


In [4]:
extra_biosphere_importer = prepare_extra_biosphere_import(
    Config.EXTRA_BIOSPHERE_PATH,
    Config.PROJECT_NAME,
)

pd.DataFrame(extra_biosphere_importer.unlinked)

Extracted 1 worksheets in 0.01 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_restore_temporal_distributions
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 15 strategies in 0.38 seconds


""


## 4. Write extra biosphere database


In [5]:
extra_biosphere_importer.write_database()

100%|██████████| 4/4 [00:00<00:00, 29330.80it/s]


15:49:51+0200 [info     ] Vacuuming database            
Created database: ecoinvent-3.12-biosphere_extra


## 5. Prepare iLUC import

The iLUC inventory is linked to consequential ecoinvent, the main biosphere database, and the extra biosphere database.


In [6]:
iluc_importer = prepare_iluc_import(
    source_path=Config.ILUC_PATH,
    project_name=Config.PROJECT_NAME,
    ecoinvent_database_name=Config.ECOINVENT_CLCA_NAME,
    biosphere_database_name=Config.ECOINVENT_BIOSPHERE_NAME,
    extra_biosphere_database_name=Config.ECOINVENT_BIOSPHERE_EXTRA_NAME,
    sheet_name=Config.ILUC_SHEET_NAME,
)

Extracted 1 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_restore_temporal_distributions
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 15 strategies in 0.59 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields

## 6. Check iLUC linking


In [7]:
pd.DataFrame(iluc_importer.unlinked)

""


## 7. Write iLUC database


In [8]:
assert iluc_importer.all_linked, (
    "Not all iLUC exchanges could be linked."
)

iluc_importer.write_database()

Graph statistics for `ecoinvent-3.12-iluc` importer:
4 graph nodes:
	None: 4
179 graph edges:
	technosphere: 147
	biosphere: 28
	production: 4
179 edges to the following databases:
	ecoinvent-3.12-consequential: 144
	ecoinvent-3.12-biosphere: 16
	ecoinvent-3.12-biosphere_extra: 12
	ecoinvent-3.12-iluc: 7
0 unique unlinked edges (0 total):




100%|██████████| 4/4 [00:00<00:00, 117.29it/s]


15:51:37+0200 [info     ] Vacuuming database            
Created database: ecoinvent-3.12-iluc


## 8. Verify the import


In [9]:
bd.databases

Databases dictionary with 10 object(s):
	BONSAI v2.4.1
	BONSAI v2.4.1 biosphere
	bafu
	biosphere3
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-biosphere_extra
	ecoinvent-3.12-consequential
	ecoinvent-3.12-iluc
	exiobase3316_extra_biosphere_b2
	exiobase3316b2